In [14]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import wandb
import torch
from accelerate.test_utils.testing import get_backend
from lightning.pytorch.loggers import WandbLogger
from lightning import Trainer
from core.data import FullBatchDataModule
from core.estimators import BiasWithMSE
from core.models import KernelRegression, NoisyMLP
from functools import partial
from lightning.pytorch import LightningModule, Trainer
from torch.func import functional_call
from core.entk import compute_empirical_ntk
device, n_devices, mem = get_backend()
torch.set_float32_matmul_precision("highest")

In [15]:
torch.manual_seed(42)
n = 100
p = 10
# model = LinearNetwork(input_dim=p, output_dim=1, hidden_dim=1000)
model = NoisyMLP(in_features=p, 
                 out_features=1, 
                 hidden_features=500, 
                 num_hidden_layers=3,
                 dropout_rate=0.,
                 loss_func=torch.nn.MSELoss(),
                 lr=1e-3)
# for param in model.parameters():
#     torch.nn.init.normal_(param, mean=0.0, std=100.)
model = model.to(device)
loss_fn = torch.nn.MSELoss()

# Data generation
X = torch.randn(n, p, device=device)
y = X @ torch.randn(p, 1, device=device) + 0.1 *torch.randn(n, 1, device=device)
dm = FullBatchDataModule(X, y, num_workers=0)


# setup for torch.func.functional_call
# detach parameters to avoid accidental backpropagation through the model
detached_params = {k: v.detach() for k, v in model.named_parameters()}
# params = dict(model.named_parameters()) # potentially uncomment if you need to backprop through the model. maybe unsafe?
def model_call(params, x):
    """ Calls the model with the given parameters and a single input x."""
    return functional_call(model, params, (x.unsqueeze(0),)).squeeze(0)


In [4]:
ntk_matrix_full = compute_empirical_ntk(
        model_call, detached_params, X, X, compute='full'
    ) # Shape: (n_1, n_2, output_dim, output_dim)
K = ntk_matrix_full.squeeze() # Shape: (n_1, n_2)
K_inv = torch.linalg.inv(K)
assert torch.allclose(K @ K_inv, torch.eye(n, device=device), atol=1e-3), "K is not invertible"
# match the initial kernel regression weights to the network's initial weights (they should make the same predictions)
init_predictions = model(X)  # Shape: (n, output_dim)
alpha_init = K_inv @ init_predictions.squeeze()  # Shape: (n, output_dim)
assert torch.allclose(init_predictions.view(-1), K @ alpha_init, atol=1e-6), "Initial predictions do not match"

/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/torch/autograd/graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [5]:
# set seed for reproducibility
torch.manual_seed(42)
alpha = torch.nn.Parameter(alpha_init.clone().detach()) # α ∈ ℝ^n
print(f'Initial norm of alpha: {alpha.norm().item()}')
alpha.register_hook(lambda grad: K_inv @ grad)  # Hook to apply natural gradient
opt = torch.optim.SGD([alpha], lr=1e-3)
steps = 1000

for i in range(steps):
    opt.zero_grad()

    pred = K @ alpha                 # f(X)
    loss = torch.nn.functional.mse_loss(pred, y.squeeze())  # L(f(X), y)
    loss.backward()                          # grad = K (Kα - y)
    with torch.no_grad():
        # This is the natural gradient step: α = K_inv @ (Kα - y)
        if i % 100 == 0:
            print(f"{30 * '='} Step {i}/{steps} {30 * '='}")
            print(f'Norm of alpha.grad: {alpha.grad.norm().item()}')
            print(f'Loss at step {i}: {loss.item()}')

        # alpha.grad.copy_(K_inv @ alpha.grad)
    opt.step()
print(f'\nFinal loss: {loss.item()}')

Initial norm of alpha: 0.03997392579913139
============================== Step 0/1000 ==============================
Norm of alpha.grad: 0.37001270055770874
Loss at step 0: 3.4227352142333984
============================== Step 100/1000 ==============================
Norm of alpha.grad: 0.33224403858184814
Loss at step 100: 2.7596523761749268
============================== Step 200/1000 ==============================
Norm of alpha.grad: 0.29935353994369507
Loss at step 200: 2.240314245223999
============================== Step 300/1000 ==============================
Norm of alpha.grad: 0.27053260803222656
Loss at step 300: 1.8296977281570435
============================== Step 400/1000 ==============================
Norm of alpha.grad: 0.24518778920173645
Loss at step 400: 1.5029263496398926
============================== Step 500/1000 ==============================
Norm of alpha.grad: 0.22283077239990234
Loss at step 500: 1.241339087486267
============================== Step 600/1000 

In [12]:
# from lightning.pytorch.loggers import CSVLogger
logger = WandbLogger(save_dir='../logs', name='nonlinear_network_ntk', project='inductive-bias')
train = Trainer(
    max_epochs=1000,
    accumulate_grad_batches=1,
    accelerator=device,
    log_every_n_steps=1,
    logger=logger,
)
train.fit(model, datamodule=dm)
wandb.finish()

/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /shared_data0/jrudoler/.cache/pypoetry/virtualenvs/i ...
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type       | Params | Mode 
-------------------------------------------------
0 | loss_func | MSELoss    | 0      | train
1 | layers    | Sequential | 507 K  | train
-------------------------------------------------
507 K     Trainable params
0         Non-trainable params
507 K     Total params
2.028     Total estimated model params size (MB)
11        Modules in train mode
0         Modules in eval mode
/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1000` reached.


epoch,▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇████
train/loss,████▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
trainer/global_step,▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
epoch,999
train/loss,0.0023
trainer/global_step,999


In [13]:
kernel_reg = KernelRegression(
    kernel = K.clone().detach(),  # Use the computed kernel matrix
    n_train_samples= n,
    ridge_lambda=0.,
    fit_intercept=False,
    init_zeros=False,
    init_weights=alpha_init.clone().detach().view(1,-1),  # Flatten the initial weights
)
kernel_reg.kernel_linear.weight.register_hook(lambda grad: (K_inv @ grad.squeeze(0)).unsqueeze(0))  # Hook to apply natural gradient

logger = WandbLogger(save_dir='../logs', name='ntk_kernel_regression', project="inductive-bias")
kernel_train = Trainer(
    max_epochs=1000,
    accumulate_grad_batches=1,
    accelerator=device,
    log_every_n_steps=1,
    logger=logger,
)
kernel_train.fit(kernel_reg, datamodule=dm)
wandb.finish()

/home/jrudoler/inductive-bias/core/models.py:264: UserWarning: Using a precomputed kernel. batch inputs will be ignored.
  warnings.warn("Using a precomputed kernel. batch inputs will be ignored.")
/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /shared_data0/jrudoler/.cache/pypoetry/virtualenvs/i ...
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type               | Params | Mode 
-------------------------------------------------------------
0 | kernel_linear | Linear             | 100    | train
1 | loss_func     | KernelRidgeMSELoss | 0      | train
-------------------------------------------------------------
100       Trainable params
0         Non-trainable params
100       Total params
0.000     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode
/shared_data0/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1000` reached.


epoch,▁▁▁▁▁▁▂▂▂▂▃▃▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
train/loss,█▅▅▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇█████
epoch,999
train/loss,0.00513
trainer/global_step,999


In [17]:
model.training_step()

TypeError: NoisyMLP.training_step() missing 2 required positional arguments: 'batch' and 'batch_idx'